In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [2]:
%%writefile .streamlit/config.toml

[theme]
base = "dark"
primaryColor = "#F4B23E"
backgroundColor = "#0D1520"
secondaryBackgroundColor = "#142030"
textColor = "#E9EEF4"
font = "sans serif"

Writing .streamlit/config.toml


In [4]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title='好球帶研究室',page_icon='⚾',layout='wide')

pages = [
    st.Page("pages/home.py", title="首頁", default=True),
    st.Page("pages/academy.py", title="數據學院"),
    st.Page("pages/watch.py", title="球季觀察室"),
    st.Page("pages/predict.py", title="台灣大賽預測"),
]

st.navigation(pages,position='top').run()

Overwriting app.py


In [5]:
%%writefile src/ui.py

"""全站共用的樣式與小元件（卡片、數字方塊、標籤）。"""
import streamlit as st

COLORS = {"bg": "#0D1520", "surface": "#142030", "surface2": "#1B2A3D", "line": "#26384F",
          "fg": "#E9EEF4", "muted": "#8C9BB0", "accent": "#F4B23E",
          "good": "#46B98C", "bad": "#E8654F", "cold": "#4A86DE", "hot": "#E2483E"}

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Barlow+Condensed:wght@600;700&family=Noto+Sans+TC:wght@400;700;900&display=swap');
html, body, [class*="css"] { font-family: 'Noto Sans TC', sans-serif; }
h1, h2, h3 { font-weight: 900 !important; letter-spacing: .02em; }
.num { font-family: 'Barlow Condensed', sans-serif; font-variant-numeric: tabular-nums; }
.eyebrow { color: #F4B23E; font-size: 13px; letter-spacing: .14em; font-weight: 700; margin: 0; }
.card { background: #142030; border: 1px solid #26384F; border-radius: 10px; padding: 18px 20px; margin-bottom: 12px; }
.kpi-label { color: #8C9BB0; font-size: 13px; }
.kpi-value { font-family: 'Barlow Condensed', sans-serif; font-size: 40px; font-weight: 700; line-height: 1.1; }
.kpi-who { font-weight: 700; }
.chip { display: inline-block; font-size: 12px; font-weight: 700; padding: 2px 9px;
        border-radius: 999px; border: 1px solid currentColor; margin-top: 4px; }
.chip.good { color: #46B98C; } .chip.bad { color: #E8654F; } .chip.warn { color: #F4B23E; }
.chip.cold { color: #4A86DE; } .chip.hot { color: #E2483E; }
.card ul { margin: 0; padding-left: 20px; } .card li { margin-bottom: 6px; }
</style>
"""

def setup(title):
    st.markdown(CSS, unsafe_allow_html=True)
    st.title(title)

def chip(text, tone="warn"):
    return f'<span class="chip {tone}">{text}</span>'

def kpi(label, value, who="", note=""):
    st.markdown(f'<div class="card"><div class="kpi-label">{label}</div>'
                f'<div class="kpi-value">{value}</div><div class="kpi-who">{who}</div>{note}</div>',
                unsafe_allow_html=True)

def bullet_card(title, items, color):
    lis = "".join(f"<li>{x}</li>" for x in items)
    st.markdown(f'<div class="card" style="border-top:4px solid {color}">'
                f'<b>{title}</b><ul style="margin-top:8px">{lis}</ul></div>', unsafe_allow_html=True)

def f3(x):
    return f"{x:.3f}".lstrip("0") if x >= 0 else "-" + f"{-x:.3f}".lstrip("0")

Writing src/ui.py


In [6]:
%%writefile src/data.py

"""網站讀資料的地方。只讀 data/clean 的 CSV，不在使用者開頁時爬網站。"""
from pathlib import Path

import pandas as pd
import streamlit as st

CLEAN = Path(__file__).resolve().parent.parent / "data" / "clean"

@st.cache_data
def load():
    teams = pd.read_csv(CLEAN / "teams.csv")
    batters = pd.read_csv(CLEAN / "batters.csv").merge(
        pd.read_csv(CLEAN / "batter_splits.csv"), left_on="player_id", right_on="batter_id", how="left")
    pitchers = pd.read_csv(CLEAN / "pitchers.csv").merge(
        pd.read_csv(CLEAN / "pitcher_splits.csv"), left_on="player_id", right_on="pitcher_id", how="left")
    batters["bats"] = batters["bats"].fillna("R")
    const = pd.read_csv(CLEAN / "league_constants.csv", index_col=0).iloc[:, 0]
    q = pitchers[pitchers["IP"] >= 30]
    lg = {"wOBA": (batters["wOBA_calc"] * batters["PA"]).sum() / batters["PA"].sum(),
          "BBp_p75": q["BBp"].quantile(0.75), "HR9_p75": q["HR9"].quantile(0.75)}
    return teams, batters, pitchers, const, lg

@st.cache_data
def load_pa():
    return pd.read_csv(CLEAN / "pa.csv", dtype={"result": str}, keep_default_na=False)

Writing src/data.py


In [7]:
%%writefile pages/home.py
import streamlit as st

from src import ui
from src.data import load

ui.setup("首頁")              # 每一頁換成自己的標題
teams, batters, pitchers, const, lg = load()
st.write(f"讀到 {len(batters)} 位打者、{len(pitchers)} 位投手。")
ui.kpi("測試卡片", ".319", "聯盟 wOBA")

Writing pages/home.py


In [8]:
%%writefile pages/academy.py
import streamlit as st
from src import ui

ui.setup("數據學院")

Writing pages/academy.py


In [9]:
%%writefile pages/watch.py
import streamlit as st
from src import ui

ui.setup("球季觀察室")

Writing pages/watch.py


In [10]:
%%writefile pages/predict.py
import streamlit as st
from src import ui

ui.setup("台灣大賽預測")

Writing pages/predict.py
